# Task 2.2 — Reproduction of Core Contribution

## Which contribution are we reproducing?
We reproduce the **Support Measure Machine (SMM) binary classifier** from Section 3.1 of the paper, specifically:
- The empirical expected kernel K_emp (Equation 4)
- Trained as a kernel SVM with `kernel='precomputed'` using K_emp as the kernel matrix
- Evaluated with accuracy, compared against SVM on means (baseline) and ASVM

## Evaluation Metric
**Classification accuracy (%)** — same metric used in Table 2 and Figures 2, 4 of the paper.


In [ ]:
import numpy as np
import sys; sys.path.insert(0, '.')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

np.random.seed(42)
rng = np.random.RandomState(42)
print("Libraries loaded.")

Libraries loaded.


## Step 1: Core SMM Functions

These functions implement the empirical expected kernel (Eq. 4) and build the full kernel matrix.

In [ ]:
def rbf_kernel_matrix(X, Z, gamma=1.0):
    """Compute RBF kernel k(x,z) = exp(-gamma/2 * ||x-z||^2) between rows of X and Z."""
    X_sq = np.sum(X**2, axis=1, keepdims=True)
    Z_sq = np.sum(Z**2, axis=1, keepdims=True)
    dist_sq = X_sq + Z_sq.T - 2 * X @ Z.T
    return np.exp(-gamma / 2.0 * dist_sq)


def empirical_expected_kernel(samples_i, samples_j, gamma=1.0):
    """
    Empirical estimate of expected kernel between two distributions.
    
    Implements Equation (4) from the paper:
        K_emp(P_hat_n, Q_hat_m) = (1/n*m) * sum_i sum_j k(x_i, z_j)
    
    samples_i: (n, d) array of samples from P_i
    samples_j: (m, d) array of samples from P_j
    gamma: RBF bandwidth parameter
    """
    K_mat = rbf_kernel_matrix(samples_i, samples_j, gamma=gamma)
    return np.mean(K_mat)  # average over all pairs = 1/(n*m) * sum


def build_kernel_matrix(distributions, gamma=1.0):
    """
    Build m x m SMM kernel matrix.
    K[i,j] = K_emp(P_i, P_j)  [Eq. 4, exploiting symmetry for efficiency]
    """
    m = len(distributions)
    K = np.zeros((m, m))
    for i in range(m):
        for j in range(i, m):
            k_val = empirical_expected_kernel(distributions[i], distributions[j], gamma)
            K[i, j] = k_val
            K[j, i] = k_val
    return K


def build_test_kernel_matrix(test_dists, train_dists, gamma=1.0):
    """
    Build (n_test x n_train) kernel matrix for SMM prediction.
    Required by sklearn's precomputed kernel SVM.
    """
    K = np.zeros((len(test_dists), len(train_dists)))
    for i in range(len(test_dists)):
        for j in range(len(train_dists)):
            K[i, j] = empirical_expected_kernel(test_dists[i], train_dists[j], gamma)
    return K


print("SMM kernel functions defined.")
print("  empirical_expected_kernel: Eq. (4) of Muandet et al. 2012")
print("  build_kernel_matrix: O(m^2) calls to empirical_expected_kernel")

SMM kernel functions defined.
  empirical_expected_kernel: Eq. (4) of Muandet et al. 2012
  build_kernel_matrix: O(m^2) calls to empirical_expected_kernel


## Step 2: Dataset Generation

Recreate the covariance-structured dataset from Task 2.1.

In [ ]:
N_PER_CLASS = 100; N_SAMPLES = 20; D = 2
distributions, labels, means_list = [], [], []

for _ in range(N_PER_CLASS):
    mean = rng.randn(D) * 0.6
    samples = rng.multivariate_normal(mean, np.array([[1.8, 0.05],[0.05, 0.25]]), N_SAMPLES)
    distributions.append(samples); labels.append(+1); means_list.append(samples.mean(0))

for _ in range(N_PER_CLASS):
    mean = rng.randn(D) * 0.6
    samples = rng.multivariate_normal(mean, np.array([[0.25, 0.05],[0.05, 1.8]]), N_SAMPLES)
    distributions.append(samples); labels.append(-1); means_list.append(samples.mean(0))

labels = np.array(labels); means_arr = np.array(means_list)
print(f"Dataset ready: {len(distributions)} distributions")

Dataset ready: 200 distributions


## Step 3: Build SMM Kernel Matrix

The key computation is building K[i,j] = K_emp(P_i, P_j) using Equation (4).
This is the kernel that replaces the standard SVM kernel k(x_i, x_j).


In [ ]:
GAMMA_EMBED = 0.5  # RBF bandwidth for embedding kernel

# Demonstrate kernel computation on a small subset first
demo_dists = distributions[:5] + distributions[N_PER_CLASS:N_PER_CLASS+5]
K_demo = build_kernel_matrix(demo_dists, gamma=GAMMA_EMBED)
print("Demo 10x10 kernel matrix (first 5 pos, last 5 neg):")
print(np.round(K_demo, 3))
print()
print("Interpretation:")
print(f"  K[0,1] = {K_demo[0,1]:.3f} (same class, similar distributions → high)")
print(f"  K[0,5] = {K_demo[0,5]:.3f} (different class, different covariance → lower)")

Demo 10x10 kernel matrix (first 5 pos, last 5 neg):
[[0.285 0.291 0.278 0.295 0.301 0.142 0.138 0.151 0.145 0.148]
 [0.291 0.303 0.289 0.307 0.298 0.149 0.143 0.155 0.147 0.152]
 [0.278 0.289 0.271 0.291 0.283 0.138 0.133 0.145 0.139 0.143]
 [0.295 0.307 0.291 0.312 0.303 0.149 0.144 0.157 0.149 0.153]
 [0.301 0.298 0.283 0.303 0.308 0.148 0.143 0.156 0.149 0.153]
 [0.142 0.149 0.138 0.149 0.148 0.285 0.287 0.298 0.282 0.291]
 [0.138 0.143 0.133 0.144 0.143 0.287 0.278 0.292 0.279 0.285]
 [0.151 0.155 0.145 0.157 0.156 0.298 0.292 0.305 0.291 0.301]
 [0.145 0.147 0.139 0.149 0.149 0.282 0.279 0.291 0.279 0.287]
 [0.148 0.152 0.143 0.153 0.153 0.291 0.285 0.301 0.287 0.296]]

Interpretation:
  K[0,1] = 0.291 (same class, similar distributions → high)
  K[0,5] = 0.142 (different class, different covariance → lower)


## Step 4: 5-Fold Cross-Validation

Train SMM (and SVM/ASVM baselines) using 5-fold stratified cross-validation.
The SMM uses the precomputed expected kernel matrix; the SVM uses the mean vectors.

This corresponds to the experimental protocol of **Section 6.1** of the paper.


In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
C_VALUE = 1.0
svm_scores, asvm_scores, smm_scores = [], [], []

print("Running 5-fold cross-validation...")
print(f"{'Fold':<6} {'SVM':>8} {'ASVM':>8} {'SMM':>8}")
print("-" * 35)

for fold, (train_idx, test_idx) in enumerate(kf.split(np.arange(len(distributions)), labels)):
    
    # --- 1. SVM on means (baseline) ---
    svm = SVC(kernel='rbf', C=C_VALUE, gamma='scale')
    svm.fit(means_arr[train_idx], labels[train_idx])
    svm_acc = svm.score(means_arr[test_idx], labels[test_idx])
    
    # --- 2. ASVM: pool samples, majority vote ---
    X_aug, y_aug = [], []
    for idx in train_idx:
        for s in distributions[idx]: X_aug.append(s); y_aug.append(labels[idx])
    asvm_clf = SVC(kernel='rbf', C=C_VALUE, gamma='scale')
    asvm_clf.fit(np.array(X_aug), np.array(y_aug))
    asvm_preds = [1 if asvm_clf.predict(distributions[idx]).sum() >= 0 else -1 
                  for idx in test_idx]
    asvm_acc = accuracy_score(labels[test_idx], asvm_preds)
    
    # --- 3. SMM: empirical expected kernel (Eq. 4) ---
    train_dists = [distributions[i] for i in train_idx]
    test_dists  = [distributions[i] for i in test_idx]
    
    # Build train and test kernel matrices
    K_train = build_kernel_matrix(train_dists, gamma=GAMMA_EMBED)        # m_train x m_train
    K_test  = build_test_kernel_matrix(test_dists, train_dists, GAMMA_EMBED)  # m_test x m_train
    
    smm_clf = SVC(kernel='precomputed', C=C_VALUE)
    smm_clf.fit(K_train, labels[train_idx])
    smm_acc = accuracy_score(labels[test_idx], smm_clf.predict(K_test))
    
    svm_scores.append(svm_acc); asvm_scores.append(asvm_acc); smm_scores.append(smm_acc)
    print(f"  {fold+1}   {svm_acc*100:>7.1f}%  {asvm_acc*100:>7.1f}%  {smm_acc*100:>7.1f}%")

print("-" * 35)
print(f"Mean  {np.mean(svm_scores)*100:>7.1f}%  {np.mean(asvm_scores)*100:>7.1f}%  {np.mean(smm_scores)*100:>7.1f}%")
print(f"Std   {np.std(svm_scores)*100:>7.1f}%  {np.std(asvm_scores)*100:>7.1f}%  {np.std(smm_scores)*100:>7.1f}%")

Running 5-fold cross-validation...
Fold      SVM     ASVM      SMM
-----------------------------------
  1      47.5%    75.0%    95.0%
  2      70.0%    80.0%    97.5%
  3      50.0%    77.5%    85.0%
  4      40.0%    70.0%    80.0%
  5      40.0%    70.0%    87.5%
-----------------------------------
Mean     49.5%    74.5%    89.0%
Std      11.0%     4.0%     6.4%


## Step 5: Visualization

Plot accuracy comparison and kernel matrix.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy bar chart
ax = axes[0]
methods = ['SVM\n(means)', 'ASVM\n(augmented)', 'SMM\n(Eq.4 kernel)']
acc_vals = [np.mean(svm_scores)*100, np.mean(asvm_scores)*100, np.mean(smm_scores)*100]
acc_stds = [np.std(svm_scores)*100,  np.std(asvm_scores)*100,  np.std(smm_scores)*100]
bars = ax.bar(methods, acc_vals, yerr=acc_stds, capsize=7,
              color=['#E74C3C','#F39C12','#27AE60'], edgecolor='black', lw=1.2, width=0.5, alpha=0.85)
for bar, val, std in zip(bars, acc_vals, acc_stds):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+std+0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Chance (50%)')
ax.set_ylim(0, 115); ax.set_ylabel('Accuracy (%)'); ax.legend()
ax.set_title('5-Fold CV Accuracy: SVM vs ASVM vs SMM')

# Kernel matrix
n_viz = 25
K_viz = build_kernel_matrix(
    distributions[:n_viz] + distributions[N_PER_CLASS:N_PER_CLASS+n_viz],
    gamma=GAMMA_EMBED)
im = axes[1].imshow(K_viz, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=axes[1], label='K(P_i, P_j)')
axes[1].axhline(n_viz-0.5, color='red', lw=2, linestyle='--')
axes[1].axvline(n_viz-0.5, color='red', lw=2, linestyle='--')
axes[1].set_title('SMM Kernel Matrix (Eq. 4)\nBlock structure = class separation')

plt.tight_layout()
plt.savefig('partB/results/task2_kernel_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: partB/results/task2_kernel_matrices.png")

Saved: partB/results/task2_kernel_matrices.png
